<a href="https://colab.research.google.com/github/julius831/-interact-decorator/blob/main/walmart_sales_projection_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project: Walmart Sales Predictio**n

## Background
Walmart faces the challenge of accurately forecasting product sales across its stores to optimize inventory management, reduce stockouts and overstock situations, and improve overall supply chain efficiency in a highly dynamic retail environment influenced by factors such as seasonality, promotions, holidays, and regional demand variations; as a Data Scientist, you are tasked with developing a robust predictive model that leverages historical sales data along with relevant features such as time-based variables, store and product attributes to generate reliable sales forecasts that can support data-driven decision-making, enhance operational planning, and ultimately improve profitability and customer satisfaction.


## About the dataset
The Walmart sales dataset is a widely used retail dataset designed for time-series forecasting and regression tasks, containing historical sales records from multiple Walmart stores across different regions. It typically includes variables such as store identifiers, department or product categories, weekly sales, and time-related features like date, along with additional attributes that capture external influences such as holidays, temperature, fuel prices, consumer price index (CPI), and unemployment rates. The dataset is structured to reflect real-world retail dynamics, where sales are affected not only by internal store operations but also by macroeconomic and seasonal factors. **bold text**






import the essential packages


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

load data set

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/BuhariS/walmart_sales_prediction/refs/heads/main/walmart.csv")

Glimpse to the data


*   EDA




In [ ]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106




* weekly sales target

*   get the **metadata**



In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB




* data small
*  Weekly sales: target
* *data type is object: drop the date, change the date then transform it.*
* no nulls










*   check for duplicates



In [ ]:
df.duplicated().sum()

np.int64(0)

***no duplicates***

***get summary statistics***

In [ ]:
df.describe()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
std,12.988182,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885
min,1.000000,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000


transform the date, or drop the date, since LR is not a classification

In [ ]:
df.drop('Date', axis=1, inplace=True)

In [ ]:
df.head()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,1554806.68,0,46.50,2.625,211.350143,8.106


***Model training***
* Extract features and targets





In [ ]:
x = df.drop('Weekly_Sales', axis=1)
y = df['Weekly_Sales']

**Split the dataset for training and testing**

In [ ]:
train_test_split(x, y, test_size=0.2, random_state=42)

[      Store  Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment
 1033      8             0        75.32       2.582  214.878556         6.315
 915       7             0        20.70       3.372  192.058484         8.818
 5903     42             0        61.24       3.130  126.546161         9.003
 2083     15             0        69.19       3.906  136.213613         7.806
 5943     42             0        87.40       3.743  129.240581         8.257
 ...     ...           ...          ...         ...         ...           ...
 3772     27             0        39.32       3.420  137.251185         7.827
 5191     37             0        54.44       2.708  210.376263         8.476
 5226     37             0        86.71       3.684  214.297294         8.177
 5390     38             1        44.64       3.428  130.071032        12.890
 860       7             0        27.28       2.550  189.534100         9.014
 
 [5148 rows x 6 columns],
       Store  Holiday_Flag  Temperat

***unpack the datasets***

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

***initialise the regressor***

In [ ]:
random_forest_regressor = RandomForestRegressor(random_state=42,n_estimators=100,max_depth=10)

***train the model***

In [ ]:
model = random_forest_regressor.fit(X_train, y_train)

***make Predictions***

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
y_pred

array([1234305.9724226 , 1313255.16696734, 1852365.26828207, ...,
        817644.79330039, 1904516.57058812,  833669.41318932])


**evaluate the performance ddof thed model**



In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")

Mean Squared Error: 23461089822.04
Root Mean Squared Error: 153170.13


In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print(f"R2 Score: {r2:.2f}")


R2 Score: 0.93


***deploying a  model***

Project deployment is the process of making a software project available for real users to access and use. In this project, we are going to use GitHub and Streamlit to deploy the product.

In [ ]:
import joblib

In [ ]:
joblib.dump(model, 'model.pkl')

['model.pkl']

In [ ]:
X_train.columns

Index(['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
       'Unemployment'],
      dtype='object')